In [4]:
from pynq import Overlay

# ---- Load overlay ----
ol = Overlay("rdo_filter_bd_wrapper.bit")   # adjust path if needed
rdo = ol.rdo_filter_axi_0

# ---- Register offsets (from AXI-Lite map) ----
CTRL         = 0x00
LAMBDA       = 0x04
R_NONE       = 0x08
R_PC         = 0x0C
R_NS         = 0x10
STATUS       = 0x14
SEL_NONE_LO  = 0x18
SEL_NONE_HI  = 0x1C
SEL_PC_LO    = 0x20
SEL_PC_HI    = 0x24
SEL_NS_LO    = 0x28
SEL_NS_HI    = 0x2C

print("Overlay + register handles ready")

Overlay + register handles ready


In [5]:
import numpy as np
from pynq import allocate

# ---- Paths ----
HR_PATH = "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_HR/frame_013.y"
SR_PATH = "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit/TEST_PRELR/RANGE_1/frame_013_qp160_prelr.yuv"

# ---- Frame geometry ----
FRAME_W, FRAME_H = 1920, 1080
RU_SIZE          = 256
NUM_RU_ROWS      = FRAME_H // RU_SIZE   # 4
NUM_RU_COLS      = FRAME_W // RU_SIZE   # 7
NUM_RU           = NUM_RU_ROWS * NUM_RU_COLS   # 28 full RUs (edges skipped for now)

# ---- Load ----
hr_frame = np.fromfile(HR_PATH, dtype=np.uint8).reshape(FRAME_H, FRAME_W)
sr_frame = np.fromfile(SR_PATH, dtype=np.uint8).reshape(FRAME_H, FRAME_W)
print(f"Loaded HR {hr_frame.shape} mean={hr_frame.mean():.1f}, SR {sr_frame.shape} mean={sr_frame.mean():.1f}")

# ---- Synthesize PC and NS candidates so different RUs get different winners ----
# PC: 50/50 blend of SR and HR (medium-quality filter)
# NS: 30/70 blend of SR and HR (higher-quality filter)
# Expectation: NS should win most RUs (lowest MSE), PC beats NONE, NONE rarely wins
pc_frame = ((sr_frame.astype(int) + hr_frame.astype(int)) // 2).astype(np.uint8)
ns_frame = ((3 * sr_frame.astype(int) + 7 * hr_frame.astype(int)) // 10).astype(np.uint8)

# ---- Register setup: lambda=0 to focus on MSE ----
rdo.write(LAMBDA, 0)
rdo.write(R_NONE, 0)
rdo.write(R_PC,   0)
rdo.write(R_NS,   0)

# ---- Allocate DMA buffers once, reuse ----
N = RU_SIZE * RU_SIZE   # 65536
hr_buf   = allocate(shape=(N,), dtype=np.uint32)
none_buf = allocate(shape=(N,), dtype=np.uint32)
pc_buf   = allocate(shape=(N,), dtype=np.uint32)
ns_buf   = allocate(shape=(N,), dtype=np.uint32)

# ---- Extract a 256x256 tile as tile-order flat array ----
def extract_ru(frame, ru_row, ru_col):
    r0, c0 = ru_row * RU_SIZE, ru_col * RU_SIZE
    return frame[r0:r0+RU_SIZE, c0:c0+RU_SIZE].flatten().astype(np.uint32)

# ---- Software golden reference: MSE + argmin with tie-break NONE > PC > NS ----
def sw_select(hr_t, none_t, pc_t, ns_t, lam=0, rn=0, rp=0, rs=0):
    diff_none = none_t.astype(int) - hr_t.astype(int)
    diff_pc   = pc_t.astype(int)   - hr_t.astype(int)
    diff_ns   = ns_t.astype(int)   - hr_t.astype(int)
    mse_none = int(np.sum(diff_none * diff_none))
    mse_pc   = int(np.sum(diff_pc   * diff_pc))
    mse_ns   = int(np.sum(diff_ns   * diff_ns))
    c_none = mse_none + lam * rn
    c_pc   = mse_pc   + lam * rp
    c_ns   = mse_ns   + lam * rs
    if c_none <= c_pc and c_none <= c_ns:  return 0, (mse_none, mse_pc, mse_ns)
    if c_pc   <= c_ns:                     return 1, (mse_none, mse_pc, mse_ns)
    return 2, (mse_none, mse_pc, mse_ns)

# ---- Clear + resync ----
rdo.write(CTRL, 0x1); rdo.write(CTRL, 0x0)
rdo.write(CTRL, 0x4); rdo.write(CTRL, 0x0)

# ---- Loop over 28 RUs, dispatch each, track SW golden ----
sw_selections = []
mse_log = []

import time
t0 = time.time()

for ru_idx in range(NUM_RU):
    ru_row = ru_idx // NUM_RU_COLS
    ru_col = ru_idx  % NUM_RU_COLS

    hr_tile   = extract_ru(hr_frame, ru_row, ru_col)
    none_tile = extract_ru(sr_frame, ru_row, ru_col)
    pc_tile   = extract_ru(pc_frame, ru_row, ru_col)
    ns_tile   = extract_ru(ns_frame, ru_row, ru_col)

    hr_buf[:]   = hr_tile
    none_buf[:] = none_tile
    pc_buf[:]   = pc_tile
    ns_buf[:]   = ns_tile

    sw_win, mses = sw_select(hr_tile, none_tile, pc_tile, ns_tile)
    sw_selections.append(sw_win)
    mse_log.append(mses)

    ol.axi_dma_0.sendchannel.transfer(hr_buf)
    ol.axi_dma_1.sendchannel.transfer(none_buf)
    ol.axi_dma_2.sendchannel.transfer(pc_buf)
    ol.axi_dma_3.sendchannel.transfer(ns_buf)
    for d in [ol.axi_dma_0, ol.axi_dma_1, ol.axi_dma_2, ol.axi_dma_3]:
        d.sendchannel.wait()

elapsed = time.time() - t0
print(f"\nProcessed {NUM_RU} RUs in {elapsed:.3f}s  ({elapsed*1000/NUM_RU:.1f} ms/RU)")

# ---- Read hardware bitmaps + status ----
sel_none = rdo.read(SEL_NONE_LO)
sel_pc   = rdo.read(SEL_PC_LO)
sel_ns   = rdo.read(SEL_NS_LO)
status   = rdo.read(STATUS)

def bits_to_winners(sn, sp, ss, n):
    out = []
    for i in range(n):
        if (sn >> i) & 1: out.append(0)
        elif (sp >> i) & 1: out.append(1)
        elif (ss >> i) & 1: out.append(2)
        else: out.append(-1)
    return out

hw_selections = bits_to_winners(sel_none, sel_pc, sel_ns, NUM_RU)

# ---- Compare per RU ----
print(f"\n{'RU':>3} {'row':>3} {'col':>3}  {'MSE_none':>10} {'MSE_pc':>10} {'MSE_ns':>10}  {'SW':>3} {'HW':>3}  Match")
mismatches = 0
for i in range(NUM_RU):
    r, c = i // NUM_RU_COLS, i % NUM_RU_COLS
    m0, m1, m2 = mse_log[i]
    match = "PASS" if hw_selections[i] == sw_selections[i] else "FAIL"
    if match == "FAIL": mismatches += 1
    print(f"{i:>3} {r:>3} {c:>3}  {m0:>10} {m1:>10} {m2:>10}   {sw_selections[i]:>1}   {hw_selections[i]:>1}   {match}")

print(f"\n{NUM_RU - mismatches}/{NUM_RU} match  ({'PASS' if mismatches == 0 else 'FAIL'})")
print(f"STATUS = 0x{status:08X}")
print(f"  num_ru_processed = {status & 0xFF}       (expect {NUM_RU})")
print(f"  ru_idx_dispatched = {(status >> 8) & 0xFF}  (expect {NUM_RU})")
print(f"  starvation_ctr    = {(status >> 16) & 0xFFFF}   (saturated=65535 is expected)")
print(f"SEL_NONE_LO = 0x{sel_none:08X}")
print(f"SEL_PC_LO   = 0x{sel_pc:08X}")
print(f"SEL_NS_LO   = 0x{sel_ns:08X}")

# ---- Optional: winner distribution ----
from collections import Counter
dist = Counter(hw_selections)
print(f"\nWinner distribution: NONE={dist.get(0,0)}  PC={dist.get(1,0)}  NS={dist.get(2,0)}")

# Cleanup
del hr_buf, none_buf, pc_buf, ns_buf

Loaded HR (1080, 1920) mean=111.5, SR (1080, 1920) mean=111.5

Processed 28 RUs in 0.848s  (30.3 ms/RU)

 RU row col    MSE_none     MSE_pc     MSE_ns   SW  HW  Match
  0   0   0     6658234    1675696     619975   2   2   PASS
  1   0   1     5664412    1422989     524666   2   2   PASS
  2   0   2     7944835    1997870     736542   2   2   PASS
  3   0   3     2076731     531484     207606   2   2   PASS
  4   0   4     2345444     593583     226577   2   2   PASS
  5   0   5     1453981     374131     148385   2   2   PASS
  6   0   6     2749628     698602     267095   2   2   PASS
  7   1   0     7302779    1830483     672980   2   2   PASS
  8   1   1     6053150    1518500     558819   2   2   PASS
  9   1   2     6674137    1672737     615832   2   2   PASS
 10   1   3     5359482    1347090     498644   2   2   PASS
 11   1   4     4264218    1072046     399481   2   2   PASS
 12   1   5     3580997     899684     336343   2   2   PASS
 13   1   6     3603998     908904     3

In [6]:
import os
import glob
import numpy as np
from pynq import allocate
import time

# ---- Paths and geometry (same as M5) ----
DATA_ROOT = "/home/xilinx/jupyter_notebooks/pc_filter/TreesAndGrass_1920_1080_30fps_8bit"
HR_DIR    = os.path.join(DATA_ROOT, "TEST_HR")
SR_DIR    = os.path.join(DATA_ROOT, "TEST_PRELR/RANGE_1")

FRAME_W, FRAME_H = 1920, 1080
RU_SIZE          = 256
NUM_RU_ROWS      = FRAME_H // RU_SIZE   # 4
NUM_RU_COLS      = FRAME_W // RU_SIZE   # 7
NUM_RU           = NUM_RU_ROWS * NUM_RU_COLS   # 28

# ---- Discover available (frame_id, qp) pairs ----
sr_files = sorted(glob.glob(os.path.join(SR_DIR, "frame_*_qp*_prelr.yuv")))
combos = []
for f in sr_files:
    base = os.path.basename(f)
    # parse "frame_013_qp160_prelr.yuv" -> ("013", "160")
    parts = base.split("_")
    frame_id = parts[1]
    qp       = parts[2].replace("qp", "")
    hr_path  = os.path.join(HR_DIR, f"frame_{frame_id}.y")
    if os.path.exists(hr_path):
        combos.append((frame_id, qp, hr_path, f))

print(f"Found {len(combos)} (frame, qp) combos to test")

# Limit to first N combos so it doesn't run forever
MAX_COMBOS = 6  # bump if you want more coverage
combos = combos[:MAX_COMBOS]

# ---- Register setup (constant across all combos) ----
rdo.write(LAMBDA, 0)
rdo.write(R_NONE, 0)
rdo.write(R_PC,   0)
rdo.write(R_NS,   0)

# ---- Allocate DMA buffers once ----
N = RU_SIZE * RU_SIZE
hr_buf   = allocate(shape=(N,), dtype=np.uint32)
none_buf = allocate(shape=(N,), dtype=np.uint32)
pc_buf   = allocate(shape=(N,), dtype=np.uint32)
ns_buf   = allocate(shape=(N,), dtype=np.uint32)

# ---- Helpers (same as M5) ----
def extract_ru(frame, ru_row, ru_col):
    r0, c0 = ru_row * RU_SIZE, ru_col * RU_SIZE
    return frame[r0:r0+RU_SIZE, c0:c0+RU_SIZE].flatten().astype(np.uint32)

def sw_select(hr_t, none_t, pc_t, ns_t):
    d0 = none_t.astype(int) - hr_t.astype(int)
    d1 = pc_t.astype(int)   - hr_t.astype(int)
    d2 = ns_t.astype(int)   - hr_t.astype(int)
    m0, m1, m2 = int(np.sum(d0*d0)), int(np.sum(d1*d1)), int(np.sum(d2*d2))
    if m0 <= m1 and m0 <= m2: return 0
    if m1 <= m2: return 1
    return 2

def bits_to_winners(sn, sp, ss, n):
    out = []
    for i in range(n):
        if   (sn >> i) & 1: out.append(0)
        elif (sp >> i) & 1: out.append(1)
        elif (ss >> i) & 1: out.append(2)
        else: out.append(-1)
    return out

def run_frame(hr_path, sr_path):
    """Run one frame's worth of RUs through the RDO. Return (match_count, total, dist)."""
    hr_frame = np.fromfile(hr_path, dtype=np.uint8).reshape(FRAME_H, FRAME_W)
    sr_frame = np.fromfile(sr_path, dtype=np.uint8).reshape(FRAME_H, FRAME_W)
    pc_frame = ((sr_frame.astype(int) + hr_frame.astype(int)) // 2).astype(np.uint8)
    ns_frame = ((3*sr_frame.astype(int) + 7*hr_frame.astype(int)) // 10).astype(np.uint8)

    rdo.write(CTRL, 0x1); rdo.write(CTRL, 0x0)   # clear
    rdo.write(CTRL, 0x4); rdo.write(CTRL, 0x0)   # sof

    sw_wins = []
    for ru_idx in range(NUM_RU):
        r, c = ru_idx // NUM_RU_COLS, ru_idx % NUM_RU_COLS
        hr_t   = extract_ru(hr_frame, r, c)
        none_t = extract_ru(sr_frame, r, c)
        pc_t   = extract_ru(pc_frame, r, c)
        ns_t   = extract_ru(ns_frame, r, c)

        hr_buf[:]   = hr_t
        none_buf[:] = none_t
        pc_buf[:]   = pc_t
        ns_buf[:]   = ns_t

        sw_wins.append(sw_select(hr_t, none_t, pc_t, ns_t))

        ol.axi_dma_0.sendchannel.transfer(hr_buf)
        ol.axi_dma_1.sendchannel.transfer(none_buf)
        ol.axi_dma_2.sendchannel.transfer(pc_buf)
        ol.axi_dma_3.sendchannel.transfer(ns_buf)
        for d in [ol.axi_dma_0, ol.axi_dma_1, ol.axi_dma_2, ol.axi_dma_3]:
            d.sendchannel.wait()

    hw_wins = bits_to_winners(rdo.read(SEL_NONE_LO),
                              rdo.read(SEL_PC_LO),
                              rdo.read(SEL_NS_LO), NUM_RU)
    matches = sum(1 for i in range(NUM_RU) if hw_wins[i] == sw_wins[i])
    from collections import Counter
    return matches, NUM_RU, Counter(hw_wins), Counter(sw_wins)

# ---- Run all combos ----
print(f"\n{'Frame':>6} {'QP':>4}  {'HW match':>10}  Winners (NONE/PC/NS)")
print("-" * 60)
all_pass = True
t0 = time.time()
for frame_id, qp, hr_path, sr_path in combos:
    matches, total, hw_dist, sw_dist = run_frame(hr_path, sr_path)
    status = "PASS" if matches == total else "FAIL"
    if matches != total: all_pass = False
    print(f"{frame_id:>6} {qp:>4}  {matches:>3}/{total:>3} {status:>4}  "
          f"HW={hw_dist.get(0,0)}/{hw_dist.get(1,0)}/{hw_dist.get(2,0)}  "
          f"SW={sw_dist.get(0,0)}/{sw_dist.get(1,0)}/{sw_dist.get(2,0)}")

elapsed = time.time() - t0
print(f"\nTotal: {'ALL PASS' if all_pass else 'SOME FAIL'}   ({elapsed:.1f}s for {len(combos)} frames)")

# Cleanup
del hr_buf, none_buf, pc_buf, ns_buf

Found 90 (frame, qp) combos to test

 Frame   QP    HW match  Winners (NONE/PC/NS)
------------------------------------------------------------
   013  160   28/ 28 PASS  HW=0/0/28  SW=0/0/28
   013  170   28/ 28 PASS  HW=0/0/28  SW=0/0/28
   013  180   28/ 28 PASS  HW=0/0/28  SW=0/0/28
   014  160   28/ 28 PASS  HW=0/0/28  SW=0/0/28
   014  170   28/ 28 PASS  HW=0/0/28  SW=0/0/28
   014  180   28/ 28 PASS  HW=0/0/28  SW=0/0/28

Total: ALL PASS   (11.2s for 6 frames)
